# 02 — Sentiment Analysis

Scores all Reddit posts with VADER and FinBERT, aggregates to daily weighted scores, and visualises sentiment time series around each event.

**Prerequisite:** Run notebook 01 first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from utils import EVENT_DATES

## 2a. Run sentiment scoring

FinBERT is slow on CPU — expect ~20 min. Runs once; re-run only if you delete processed files.

In [ ]:
import sentiment
sentiment.run()

## 2b. Load scores

In [ ]:
scores = pd.read_csv('../data/processed/sentiment_scores.csv', parse_dates=['date'])
scores['date'] = pd.to_datetime(scores['date']).dt.date
print(scores.groupby('event')[['vader_weighted', 'finbert_weighted']].describe().round(3))

## 2c. Sentiment time series — VADER vs FinBERT

In [ ]:
events = list(EVENT_DATES.keys())
fig, axes = plt.subplots(4, 1, figsize=(12, 16))

for ax, event in zip(axes, events):
    df = scores[scores['event'] == event].copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    event_dt = pd.Timestamp(EVENT_DATES[event])

    ax.plot(df['date'], df['vader_weighted'],   label='VADER',   color='#4A90D9', linewidth=1.8)
    ax.plot(df['date'], df['finbert_weighted'], label='FinBERT', color='#B5933A', linewidth=1.8, linestyle='--')
    ax.axvline(event_dt, color='red', linestyle=':', linewidth=1.5, label='Event')
    ax.axhline(0, color='black', linewidth=0.5, alpha=0.4)
    ax.set_title(event.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel('Weighted Sentiment')
    ax.legend(loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Sentiment Time Series by Event — VADER vs FinBERT', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/sentiment_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2d. VADER vs FinBERT correlation

How often do the two models agree?

In [ ]:
from scipy.stats import pearsonr, spearmanr

r_p, p_p = pearsonr(scores['vader_weighted'].dropna(), scores['finbert_weighted'].dropna())
r_s, p_s = spearmanr(scores['vader_weighted'].dropna(), scores['finbert_weighted'].dropna())

print(f'Pearson r  = {r_p:.3f}  (p={p_p:.4f})')
print(f'Spearman r = {r_s:.3f}  (p={p_s:.4f})')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(scores['vader_weighted'], scores['finbert_weighted'], alpha=0.5, color='#B5933A', s=30)
ax.set_xlabel('VADER Score')
ax.set_ylabel('FinBERT Score')
ax.set_title(f'VADER vs FinBERT Agreement\nPearson r={r_p:.3f}, Spearman r={r_s:.3f}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.savefig('../results/figures/vader_vs_finbert.png', dpi=150, bbox_inches='tight')
plt.show()

## 2e. Per-subreddit sentiment breakdown

Do finance subreddits skew differently from fashion subreddits?

In [ ]:
from pathlib import Path

all_scored = []
for event in EVENT_DATES:
    p = Path(f'../data/processed/{event}_scored_posts.csv')
    if p.exists():
        df = pd.read_csv(p)
        df['event'] = event
        all_scored.append(df)

if all_scored:
    posts = pd.concat(all_scored, ignore_index=True)

    sub_sentiment = posts.groupby('subreddit')[['vader', 'finbert']].mean().round(3)
    sub_sentiment['n_posts'] = posts.groupby('subreddit').size()
    print(sub_sentiment.sort_values('vader', ascending=False))